In [3]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Load Data
data_fake = pd.read_csv(r'C:\Users\91965\Downloads\Fake (1).csv')
data_true = pd.read_csv(r'C:\Users\91965\Downloads\True (1).csv')

# Labeling and Combining
data_true['label'] = 1
data_fake['label'] = 0
news = pd.concat([data_true, data_fake], axis=0)

# Drop Unnecessary Columns
news = news.drop(['text', 'subject', 'date'], axis=1)

# Shuffle and Reset Index
news = news.sample(frac=1).reset_index(drop=True)

# Rename Column
news = news.rename(columns={'title': 'text'})

# Text Preprocessing
def wordopt(text):
    text = text.lower()
    text = re.sub('\[.*?\]', ' ', text)
    text = re.sub("\\W", " ", text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

news['text'] = news['text'].apply(wordopt)

# Split Data
x = news['text']
y = news['label']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Vectorization
vectorization = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
x_train_list = x_train.astype(str).tolist()
x_test_list = x_test.astype(str).tolist()
xv_train = vectorization.fit_transform(x_train_list)
xv_test = vectorization.transform(x_test_list)

# Logistic Regression with Hyperparameter Tuning
param_grid_lr = {'C': [0.1, 1, 10]}
grid_search_lr = GridSearchCV(LogisticRegression(), param_grid_lr, cv=5)
grid_search_lr.fit(xv_train, y_train)
print("Best parameters for Logistic Regression:", grid_search_lr.best_params_)
pred_lr = grid_search_lr.predict(xv_test)
print("Logistic Regression Classification Report:\n", classification_report(y_test, pred_lr))

# Decision Tree Classifier with Cross-Validation
dtc = DecisionTreeClassifier()
dtc.fit(xv_train, y_train)
pred_dtc = dtc.predict(xv_test)
print("Decision Tree Classification Report:\n", classification_report(y_test, pred_dtc))

# Random Forest Classifier with Cross-Validation
rfc = RandomForestClassifier()
rfc.fit(xv_train, y_train)
pred_rfc = rfc.predict(xv_test)
print("Random Forest Classification Report:\n", classification_report(y_test, pred_rfc))

# Gradient Boosting Classifier with Cross-Validation
gbc = GradientBoostingClassifier()
gbc.fit(xv_train, y_train)
pred_gbc = gbc.predict(xv_test)
print("Gradient Boosting Classification Report:\n", classification_report(y_test, pred_gbc))

# Confusion Matrices
print("Confusion Matrix for Logistic Regression:\n", confusion_matrix(y_test, pred_lr))
print("Confusion Matrix for Decision Tree Classifier:\n", confusion_matrix(y_test, pred_dtc))
print("Confusion Matrix for Random Forest Classifier:\n", confusion_matrix(y_test, pred_rfc))
print("Confusion Matrix for Gradient Boosting Classifier:\n", confusion_matrix(y_test, pred_gbc))

# Function for Manual Testing
def output_label(n):
    return "It is Fake News" if n == 0 else "It is True News"

def manual_testing(news):
    testing_news = {"text": [news]}
    news_def_test = pd.DataFrame(testing_news)
    new_x_test = news_def_test["text"].apply(wordopt)
    new_xv_test = vectorization.transform(new_x_test)
    pred_lr = grid_search_lr.predict(new_xv_test)
    pred_rfc = rfc.predict(new_xv_test)
    pred_dtc = dtc.predict(new_xv_test)
    pred_gbc = gbc.predict(new_xv_test)
    return "\n\nLR Prediction: {}  \nGBC Prediction: {}  \nRFC Prediction: {}  \nDTC Prediction: {}".format(
        output_label(pred_lr[0]), output_label(pred_gbc[0]), output_label(pred_rfc[0]), output_label(pred_dtc[0])
    )

# Example Manual Testing
'''print(manual_testing("The Pentagon is considering a Boeing proposal to supply Ukraine with cheap, small precision bombs fit ted on to abundantly available rockets, allowing Kyiv to strike far behind Russian lines, according t o a Reuters report. US and allied military inventories are shrinking, and Ukraine faces an increasin g need for more sophisticated weapons as the war drags on. Boeing's proposed system, dubnd-La unched Small Diameter Bomb (GLSDB), is one of about a half-dozen plans for getting new munitions into production for Ukraine and America's eastern European allies, industry sources told the news agency. GLSDB could be delivered as early as spring 2023, according to a document reviewed by Reuters and thr ee people familiar with the plan. It combines the GBU-39 Small Diameter Bomb (SDB) with the M26 rocke t motor, both of which are common in US inventories. Although a handful of GLSDB units have already been made, there are many logistical obstacles to formal procurement. The Boeing plan requires a pric e discovery waiver, exempting the contractor from an in-depth review that ensures the Pentagon is get ting the best deal possible. Any arrangement would also require at least six suppliers to expedite sh ipments of their parts and services to produce the weapon quickly. Although the US has rebuffed requ ests for the 185-mile (297km) range Atacms missile, the GLSDB's 94-mile (150km) range would allow Ukr aine to hit valuable military targets that have been out of reach and help it continue pressing its c ounterattacks by disrupting Russian rear areas. GLSDB is made jointly by Saab AB and Boeing Co and h as been in development since 2019, well before the invasion, which Russia calls a special operatio n. In October, SAAB chief executive Micael Johansson said of the GLSDB: We are imminently shortly e xpecting contracts on that. According to the document a Boeing proposal to US European has smal 1, folding wings that allow it to glide more than 100km if dropped from an aircraft and targets as sm all as 3ft in diameter."))'''
print(manual_testing("Salman Khan married Aishwarya Rai"))




Best parameters for Logistic Regression: {'C': 10}
Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.94      0.95      4707
           1       0.94      0.94      0.94      4273

    accuracy                           0.94      8980
   macro avg       0.94      0.94      0.94      8980
weighted avg       0.94      0.94      0.94      8980

Decision Tree Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.91      0.90      4707
           1       0.90      0.87      0.88      4273

    accuracy                           0.89      8980
   macro avg       0.89      0.89      0.89      8980
weighted avg       0.89      0.89      0.89      8980

Random Forest Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.93      0.93      4707
           1       0.92      0.92      0.92      4273

    accurac

In [4]:
import pickle

# Save the models to disk
with open('model_lr.pkl', 'wb') as file_lr:
    pickle.dump(grid_search_lr.best_estimator_, file_lr)  # Save Logistic Regression model

with open('model_rfc.pkl', 'wb') as file_rfc:
    pickle.dump(rfc, file_rfc)  # Save Random Forest Classifier model

with open('model_dtc.pkl', 'wb') as file_dtc:
    pickle.dump(dtc, file_dtc)  # Save Decision Tree Classifier model

with open('model_gbc.pkl', 'wb') as file_gbc:
    pickle.dump(gbc, file_gbc)  # Save Gradient Boosting Classifier model


In [5]:
# Load the models from disk
with open('model_lr.pkl', 'rb') as file_lr:
    loaded_lr = pickle.load(file_lr)

with open('model_rfc.pkl', 'rb') as file_rfc:
    loaded_rfc = pickle.load(file_rfc)

with open('model_dtc.pkl', 'rb') as file_dtc:
    loaded_dtc = pickle.load(file_dtc)

with open('model_gbc.pkl', 'rb') as file_gbc:
    loaded_gbc = pickle.load(file_gbc)

# Save the vectorizer
with open('vectorizer.pkl', 'wb') as file_vec:
    pickle.dump(vectorization, file_vec)

